<a href="https://colab.research.google.com/github/KPhanindraReddy/chat_bot_mental_health/blob/main/chat_bot_mental_health.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio transformers huggingface_hub openvino openvino-genai py-cpuinfo


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 12.5 MB/s eta 0:00:00


In [2]:
import os
import gc
import cpuinfo
import gradio as gr
from queue import Queue, Empty
from threading import Event, Lock
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import snapshot_download
import openvino_genai
from transformers import pipeline


class ChatbotSystem:
    def __init__(self):
        self.pipe_lock = Lock()
        self.mistral_pipe = None
        self.generation_executor = ThreadPoolExecutor(max_workers=1)

        # Initialize OpenVINO Mistral
        self.initialize_model()

        # Sentiment model
        self.sentiment_analyzer = pipeline("sentiment-analysis")

    def initialize_model(self):
        """Download and initialize Mistral OpenVINO model"""
        if not os.path.exists("mistral-ov"):
            snapshot_download(
                repo_id="OpenVINO/mistral-7b-instruct-v0.1-int8-ov",
                local_dir="mistral-ov"
            )

        cpu_features = cpuinfo.get_cpu_info().get("flags", [])
        config_options = {}

        if "avx512" in cpu_features:
            config_options["ENFORCE_BF16"] = "YES"
        elif "avx2" in cpu_features:
            config_options["INFERENCE_PRECISION_HINT"] = "f32"

        self.mistral_pipe = openvino_genai.LLMPipeline(
            "mistral-ov",
            device="CPU",
            config={
                "PERFORMANCE_HINT": "THROUGHPUT",
                **config_options
            }
        )

    def analyze_sentiment(self, text: str) -> str:
        result = self.sentiment_analyzer(text[:512])[0]
        label, score = result["label"], result["score"]

        if label == "NEGATIVE" and score > 0.7:
            return "It sounds like you’re going through something difficult. You’re not alone 💙"
        elif label == "POSITIVE":
            return "I’m glad to hear that! Keep going 🌟"
        else:
            return "I understand. Feel free to share more."

    def generate_text_stream(self, prompt: str, max_tokens: int):
        response_queue = Queue()
        completion_event = Event()
        error = [None]

        config = openvino_genai.GenerationConfig(
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            streaming=True,
            streaming_interval=5
        )

        def callback(tokens):
            response_queue.put("".join(tokens))
            return openvino_genai.StreamingStatus.RUNNING

        def generate():
            try:
                with self.pipe_lock:
                    self.mistral_pipe.generate(prompt, config, callback)
            except Exception as e:
                error[0] = str(e)
            finally:
                completion_event.set()

        self.generation_executor.submit(generate)

        accumulated = []
        while not completion_event.is_set() or not response_queue.empty():
            if error[0]:
                yield f"❌ Error: {error[0]}"
                return
            try:
                chunk = response_queue.get(timeout=0.1)
                accumulated.append(chunk)
                yield "".join(accumulated)
                gc.collect()
            except Empty:
                continue

        yield "".join(accumulated)


# ------------------------
# Gradio UI
# ------------------------
chatbot_system = ChatbotSystem()

with gr.Blocks(title="Chatbot For Mental Health Using NLP") as demo:
    gr.Markdown("# 🧠 Mental Health Chatbot (CSE-35 Team)")

    chatbot = gr.Chatbot(height=500)
    user_input = gr.Textbox(label="Your Message", placeholder="Type your message...")
    max_tokens = gr.Slider(50, 1024, value=256, step=50, label="Max Tokens")
    send_btn = gr.Button("Send", variant="primary")

    def respond(message, history, max_tokens):
        if not message.strip():
            return history, ""

        support_msg = chatbot_system.analyze_sentiment(message)
        history = history + [[message, support_msg]]

        for chunk in chatbot_system.generate_text_stream(message, max_tokens):
            history[-1][1] = chunk
            yield history, ""

    send_btn.click(
        respond,
        inputs=[user_input, chatbot, max_tokens],
        outputs=[chatbot, user_input]
    )

    user_input.submit(
        respond,
        inputs=[user_input, chatbot, max_tokens],
        outputs=[chatbot, user_input]
    )

demo.launch(share=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

openvino_detokenizer.bin:   0%|          | 0.00/493k [00:00<?, ?B/s]

openvino_model.bin:   0%|          | 0.00/7.28G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu
/tmp/ipython-input-3508002681.py:113: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=500)
/tmp/ipython-input-3508002681.py:113: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=500)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6e679b76e5ba6f342.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
